# Fichier `src/cesipath/__init__.py`

Ce notebook liste tout ce que le package `cesipath` expose en import direct. Il sert de carte d'entree pour un developpeur qui debarque sur le projet.

## 1. Regle d'architecture

`__init__.py` re-exporte les objets metier **sans** effet de bord. En particulier :

- pas d'import de `matplotlib` au niveau du package ;
- pas d'I/O, pas de chargement de donnees ;
- pas de tirage aleatoire au niveau du module.

Cela rend `import cesipath` instantane, meme sur un environnement qui n'a pas matplotlib installe.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import cesipath
print("Nombre d'exports :", len(cesipath.__all__))

## 2. Les primitives graphe

Tout ce qui concerne la construction et la manipulation d'un graphe statique :

- `GraphGenerationConfig` : parametres ;
- `GraphGenerator` : generateur d'instances ;
- `GraphInstance` : instance complete ;
- `EdgeAttributes`, `EdgeStatus` : description d'une arete ;
- `InstanceValidator` : validation statique.

In [ ]:
from cesipath import (
    GraphGenerationConfig,
    GraphGenerator,
    GraphInstance,
    EdgeAttributes,
    EdgeStatus,
    InstanceValidator,
)

cfg = GraphGenerationConfig(node_count=6, seed=42)
instance = GraphGenerator(cfg).generate()
print("Valide ?", InstanceValidator(cfg).is_valid(instance))
print("Sommets :", instance.node_count)

## 3. Fermeture metrique

Les primitives Dijkstra et la verification metrique :

- `build_cost_matrix` : dict -> matrice ;
- `complete_graph_with_shortest_paths` : Dijkstra depuis chaque source ;
- `dijkstra` : version bas-niveau ;
- `check_triangle_inequality` : garde-fou.

In [ ]:
from cesipath import (
    build_cost_matrix,
    check_triangle_inequality,
    complete_graph_with_shortest_paths,
    dijkstra,
)

ok, _ = check_triangle_inequality(instance.completed_costs)
print("Matrice metrique ?", ok)

## 4. Simulation dynamique

Toute la couche dynamique :

- `DynamicNetworkSimulator` : fait evoluer le reseau ;
- `DynamicGraphSnapshot` : etat a un tour ;
- `DynamicStateValidator` : invariants dynamiques ;
- `sample_dynamic_edge_cost`, `initialize_dynamic_edge_costs`, `refresh_dynamic_edge_costs`, `dynamic_multiplier`, `DEFAULT_DYNAMIC_SIGMA` : primitives de cout.

In [ ]:
from cesipath import (
    DEFAULT_DYNAMIC_SIGMA,
    DynamicGraphSnapshot,
    DynamicNetworkSimulator,
    DynamicStateValidator,
)

simulator = DynamicNetworkSimulator(instance, seed=42)
snap = simulator.initialize_snapshot()
print("sigma par defaut :", DEFAULT_DYNAMIC_SIGMA)
print("Aretes actives   :", snap.active_edge_count)

## 5. Contrat d'entree solveur

Le seul contrat consomme par les algorithmes :

- `SolverInput` : dataclass frozen ;
- `build_static_solver_input`, `build_dynamic_solver_input` : fabriques.

In [ ]:
from cesipath import SolverInput, build_static_solver_input

solver_input = build_static_solver_input(instance)
print(type(solver_input).__name__, solver_input.source)

## 6. Algorithmes (metaheuristiques)

Quatre algorithmes partageant la meme signature `algo(solver_input, **kwargs) -> VRPSolution` :

- `grasp` : construction gloutonne randomisee + recherche locale ;
- `simulated_annealing` : Metropolis + refroidissement geometrique ;
- `tabu_search` : memoire courte + aspiration ;
- `genetic_algorithm` : giant-tour + Split de Prins + OX crossover.

In [ ]:
from cesipath import (
    VRPSolution,
    grasp,
    simulated_annealing,
    tabu_search,
    genetic_algorithm,
)

solution = grasp(solver_input, max_iterations=5, rcl_alpha=0.2, seed=42)
print("Type      :", type(solution).__name__)
print("Tournees  :", solution.route_count)
print("Cout total:", round(solution.total_cost, 2))

## 7. Benchmarks

Outillage pour comparer les algorithmes sur un lot d'instances :

- `run_benchmark(sizes, seeds, algos, algo_kwargs)` ;
- `summarize_benchmark(rows)` ;
- `plot_benchmark_quality` / `plot_benchmark_gap` / `plot_benchmark_runtime` ;
- `save_benchmark_figures(rows)` : sauvegarde les 3 figures en PNG.

Voir `benchmark.ipynb` pour un exemple complet.

In [ ]:
from cesipath import run_benchmark, summarize_benchmark

rows = run_benchmark(
    sizes=[6, 8],
    seeds=[1, 2],
    algos={"grasp": grasp, "tabu": tabu_search},
    algo_kwargs={
        "grasp": {"max_iterations": 3, "rcl_alpha": 0.2},
        "tabu":  {"max_iterations": 30, "tabu_tenure": 5},
    },
    verbose=False,
)
for s in summarize_benchmark(rows):
    print(s)

## 8. Branche dynamique

Execution et benchmark d'algorithmes sur scenarios dynamiques :

- `DynamicExecution`, `execute_dynamic` : rejoue une solution sur un reseau dynamique et mesure l'ecart planifie / realise ;
- `run_dynamic_benchmark`, `summarize_dynamic_benchmark` ;
- `plot_dynamic_cost_comparison`, `plot_dynamic_gain`, `plot_dynamic_planned_vs_realized`, `save_dynamic_benchmark_figures`.

In [ ]:
from cesipath import DynamicExecution, execute_dynamic

print("DynamicExecution dispo :", DynamicExecution.__name__)
print("execute_dynamic dispo  :", execute_dynamic.__name__)

## 9. Visualisation algorithmique

Re-exportee depuis `cesipath.algorithms.visualization` :

- `plot_solution(instance, solution)` ;
- `save_solution_plot(...)` : sauvegarde avec numero auto-incremente.

La visualisation du graphe (sans solution) reste dans `cesipath.visualization` et n'est pas re-exportee pour eviter un import matplotlib automatique.

In [ ]:
from cesipath import plot_solution
print("plot_solution dispo :", plot_solution.__name__)

## 10. Resume : ce qu'il faut importer selon le besoin

| Je veux... | J'importe... |
|---|---|
| Generer une instance | `GraphGenerationConfig`, `GraphGenerator` |
| Simuler la dynamique | `DynamicNetworkSimulator` |
| Preparer un solveur | `build_static_solver_input` ou `build_dynamic_solver_input` |
| Resoudre | `grasp` / `simulated_annealing` / `tabu_search` / `genetic_algorithm` |
| Comparer des algos | `run_benchmark`, `plot_benchmark_*` |
| Visualiser une tournee | `plot_solution`, `save_solution_plot` |
| Visualiser un graphe | `from cesipath.visualization import GraphVisualizer` |